# Descente de gradient projetée

Ce TP utilisera les bibliothèques `numpy`, `matplotlib.pyplot` et `time` qui sont importées de cette façon:

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import time

Dans les TP précédents, nous avons minimisé des fonctions différentiables sur $\mathbb{R}^n$ tout entier. Dans de nombreux problèmes, l'inconnue doit cependant vérifier des **contraintes**. Dans ce TP, nous considérons des problèmes de la forme
$$
    \min_{x \in C} f(x) \tag{P}
$$
où
- $f : \mathbb{R}^n \to \mathbb{R}$ est convexe, différentiable, et son gradient $\nabla f$ est $L$-Lipschitz,
- $C \subset \mathbb{R}^n$ est un ensemble convexe, fermé et non vide.

L'**algorithme du gradient projeté** consiste à faire un pas de gradient, puis à ramener l'itéré dans $C$ :
$$
    x_{k+1} = P_C\big(x_k - s \nabla f(x_k)\big), \qquad \textrm{où } P_C(x) = \underset{z \in C}{\textrm{argmin}} \ \| z - x \|_2
$$
est la projection orthogonale sur $C$. Cet algorithme n'est intéressant en pratique que si $P_C$ se calcule facilement.

Le TP est organisé en trois parties :
1. projections sur des ensembles simples et gradient projeté en dimension 2,
2. application à l'**inpainting** (reconstruction de pixels manquants),
3. application au **débruitage sous contrainte**.

## I. Projections et gradient projeté en dimension 2

### 1. Projections sur des ensembles simples

Nous considérons deux ensembles convexes fermés de $\mathbb{R}^n$ :
- le **pavé** $C = \{ x \in \mathbb{R}^n : a_i \leq x_i \leq b_i, \ i = 1, \dots, n\}$ où $a, b \in \mathbb{R}^n$ avec $a \leq b$,
- la **boule** $B(c, r) = \{ x \in \mathbb{R}^n : \|x - c\|_2 \leq r\}$ où $c \in \mathbb{R}^n$ et $r > 0$.

**Todo (théorie) :** Montrer que
$$
    P_{C}(x)_i = \min\big(\max(x_i, a_i), b_i\big), \qquad i = 1, \dots, n,
$$
et que
$$
    P_{B(c,r)}(x) = \left\{
        \begin{split}
            x & \quad \textrm{si } \|x - c\|_2 \leq r, \\
            c + r \frac{x - c}{\|x - c\|_2} & \quad \textrm{sinon.}
        \end{split}
    \right.
$$

**Todo :** Coder les fonctions `proj_box(x, a, b)` et `proj_ball(x, c, r)`. Ces fonctions doivent fonctionner pour des `np.array` de forme quelconque (vecteurs de $\mathbb{R}^2$, mais aussi images de taille $M \times N$ dans les parties suivantes). On pourra utiliser `np.clip` ou `np.minimum` et `np.maximum`. Attention, `np.linalg.norm` appliqué à une matrice ne calcule pas la norme euclidienne de l'image vue comme un vecteur de $\mathbb{R}^{MN}$ : préférer `np.sqrt(np.sum(...**2))`.

In [ ]:
def proj_box(x, a, b):
    ### à coder

def proj_ball(x, c, r):
    ### à coder

### 2. Test des projections

Rappelons la caractérisation de la projection sur un convexe fermé $C$ :
$$
    p = P_C(x) \iff p \in C \quad \textrm{et} \quad \langle x - p, z - p \rangle \leq 0, \quad \forall z \in C.
$$
Géométriquement, l'angle entre $x - p$ et $z - p$ est obtus pour tout $z \in C$.

Comme pour le test des adjoints du TP1, cette propriété permet de tester une implémentation : on tire au hasard des points $x$ et des points $z \in C$ et on vérifie que $\langle x - P_C(x), z - P_C(x) \rangle$ est négatif (ou de l'ordre de la précision machine). On vérifie aussi que $P_C(P_C(x)) = P_C(x)$.

La fonction suivante réalise ces tests. Pour générer un point $z \in C$, elle projette simplement un point aléatoire sur $C$.

In [ ]:
def test_projection(proj, shape, n_test=1000):
    worst_angle = -np.inf
    worst_idem = 0
    for k in range(n_test):
        x = 3 * np.random.randn(*shape)
        z = proj(3 * np.random.randn(*shape))  # un point de C
        p = proj(x)
        worst_angle = max(worst_angle, np.sum((x - p) * (z - p)))
        worst_idem = max(worst_idem, np.sqrt(np.sum((proj(p) - p)**2)))
    print('max <x - P(x), z - P(x)> : %e  (doit être <= 0 ou de l\'ordre de 1e-15)' % worst_angle)
    print('max ||P(P(x)) - P(x)||   : %e  (doit être de l\'ordre de 1e-15)' % worst_idem)

**Todo :** Tester les fonctions `proj_box` avec $a = (-1, -1)$, $b = (1, 1)$ (variables `a_box` et `b_box`, le nom `b` étant utilisé plus loin) et `proj_ball` avec $c = (0, 0)$, $r = 1$.

In [ ]:
a_box = np.array([-1., -1.])
b_box = np.array([1., 1.])
c = np.array([0., 0.])
r = 1.

### à coder

**Todo :** Appliquer le test à la fonction `proj_sphere` ci-dessous, qui ramène tout point sur la sphère $S(c, r) = \{x : \|x - c\| = r\}$. Que constatez-vous ? Expliquer.

In [ ]:
def proj_sphere(x, c, r):
    d = x - c
    return c + r * d / np.sqrt(np.sum(d**2))

### à coder

### 3. Gradient projeté sur un exemple en dimension 2

Considérons la fonction quadratique
$$
    f(x) = \frac{1}{2} \langle A x, x \rangle - \langle b, x \rangle, \qquad A = \begin{pmatrix} 3 & 0.5 \\ 0.5 & 1 \end{pmatrix}, \quad b = A \begin{pmatrix} 2 \\ 0 \end{pmatrix}.
$$
Son minimiseur sur $\mathbb{R}^2$ est $\bar{x} = (2, 0)$. Nous allons la minimiser sur le pavé $[-1,1]^2$ et sur la boule unité $B(0, 1)$, qui ne contiennent pas $\bar{x}$.

**Todo :** Donner l'expression de $\nabla f$ et de sa constante de Lipschitz $L$. Compléter le code suivant.

In [ ]:
A = np.array([[3., 0.5], [0.5, 1.]])
x_bar = np.array([2., 0.])
b = A @ x_bar

def f(x):
    return 0.5 * x @ A @ x - b @ x

def grad_f(x):
    ### à coder

L = ### à coder

**Todo :** Coder une fonction `gradient_projete_2d(grad_f, proj, x0, step, nit)` qui effectue `nit` itérations du gradient projeté à pas fixe `step` en partant de $P_C(x_0)$, et qui renvoie un `np.array` de taille `(nit+1, 2)` contenant les itérés.

In [ ]:
def gradient_projete_2d(grad_f, proj, x0, step, nit):
    ### à coder

La fonction suivante affiche les lignes de niveau de $f$, l'ensemble des contraintes et les itérés.

In [ ]:
def plot_trajectory(f, X, ax=None, title='', box=None, ball=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(5, 5))
    t = np.linspace(-2.5, 2.5, 150)
    T1, T2 = np.meshgrid(t, t)
    Z = np.array([[f(np.array([t1, t2])) for t1, t2 in zip(r1, r2)] for r1, r2 in zip(T1, T2)])
    ax.contour(T1, T2, Z, levels=30, cmap='viridis', linewidths=0.7)
    if box is not None:
        lo, hi = box
        ax.add_patch(plt.Rectangle(lo, hi[0] - lo[0], hi[1] - lo[1], fill=False, ec='k', lw=2))
    if ball is not None:
        c, r = ball
        ax.add_patch(plt.Circle(c, r, fill=False, ec='k', lw=2))
    ax.plot(X[:, 0], X[:, 1], 'o-', color='tab:red', ms=3)
    ax.plot(*x_bar, 'k*', ms=12)
    ax.set_aspect('equal')
    ax.set_title(title)
    return ax

**Todo :** Appliquer le gradient projeté avec le pas $s = 1/L$, $x_0 = (-2, 2)$ et 50 itérations, sur le pavé $[-1,1]^2$ puis sur la boule $B(0,1)$. Afficher les trajectoires. Comment se comportent les itérés lorsqu'ils atteignent le bord de $C$ ?

In [ ]:
x0 = np.array([-2., 2.])
step = 1 / L
nit = 50

### à coder

## II. Inpainting

L'**inpainting** consiste à reconstruire les pixels manquants d'une image : pixels défectueux d'un capteur, rayures sur une photographie, pertes lors d'une transmission, objet à effacer...

Notons $\Omega$ l'ensemble des pixels connus et $m$ le masque associé : $m[i,j] = 1$ si $(i,j) \in \Omega$ et $m[i,j] = 0$ sinon. On observe
$$
    v = m \odot u
$$
où $\odot$ désigne le produit terme à terme (les pixels manquants valent 0).

Nous n'avons aucune information sur les pixels manquants. Nous allons donc chercher, parmi toutes les images qui coïncident avec $v$ sur $\Omega$, la plus « régulière » au sens de la régularisation $R$ du TP1 :
$$
    \min_{u \in C_\Omega} R(u), \qquad C_\Omega = \big\{ u \in \mathbb{R}^{M \times N} : u[i,j] = v[i,j], \ \forall (i,j) \in \Omega \big\},
$$
avec
$$
    R(u) = \sum_{i,j} \sqrt{\epsilon^2 + |Du|^2[i,j]}.
$$

### 1. Données et fonctions du TP1

Les fonctions suivantes ont été codées au TP1.

In [ ]:
u = plt.imread('image.jpg') # loading image, values are integers in [0,255]
u = np.mean(np.double(u) / 255, axis = 2) # converting to grayscale and double precision in [0,1].
M, N = u.shape

def psnr(u,v):
    return -10 * np.log10(np.mean((u - v) ** 2))

def d1(u):
    du=np.zeros(u.shape)
    du[:-1,:] = u[1:,:] - u[:-1,:]
    return du

def d2(u):
    du=np.zeros(u.shape)
    du[:,:-1] = u[:,1:] - u[:,:-1]
    return du

def d1T(u):
    du=np.zeros(u.shape)
    du[0,:] = -u[0,:]
    du[1:-1,:] = -u[1:-1,:] + u[:-2,:]
    du[-1,:] = u[-2,:]
    return du

def d2T(u):
    du=np.zeros(u.shape)
    du[:,0] = -u[:,0]
    du[:,1:-1] = -u[:,1:-1] + u[:,:-2]
    du[:,-1] = u[:,-2]
    return du

**Todo :** Coder les fonctions `R(x, epsilon)` et `grad_R(x, epsilon)` qui calculent $R(x)$ et
$$
    \nabla R(x) = D_1^T  \left( \frac{D_1 x}{{\sqrt{\epsilon^2 + |Dx|^2}}}   \right) + D_2^T  \left( \frac{D_2 x}{{\sqrt{\epsilon^2 + |Dx|^2}}}   \right).
$$

In [ ]:
def R(x, epsilon):
    ### à coder

def grad_R(x, epsilon):
    ### à coder

Le code suivant génère un masque où une proportion `p` des pixels, choisis aléatoirement, est manquante, puis affiche l'image observée $v$.

In [ ]:
np.random.seed(0)
p = 0.5
mask = np.random.rand(M, N) > p    # True si le pixel est connu
v = mask * u

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
axs[0].imshow(u, cmap='gray'), axs[0].set_title('u')
axs[1].imshow(v, cmap='gray'), axs[1].set_title('v (%d%% de pixels manquants)' % (100 * p))
plt.show()

### 2. Projection sur $C_\Omega$

**Todo (théorie) :** Montrer que $C_\Omega$ est un convexe fermé non vide et que
$$
    P_{C_\Omega}(x)[i,j] = \left\{
        \begin{split}
            v[i,j] & \quad \textrm{si } (i,j) \in \Omega, \\
            x[i,j] & \quad \textrm{sinon.}
        \end{split}
    \right.
$$

**Todo :** Coder la fonction `proj_inpainting(x, mask, v)` (on pourra utiliser `np.where`) et la tester avec `test_projection`.

In [ ]:
def proj_inpainting(x, mask, v):
    ### à coder

### à coder

### 3. Gradient projeté

La constante de Lipschitz de $\nabla R$ est $L = \frac{8}{\epsilon}$ (voir TP1). Nous utiliserons le pas $s = 1/L$.

**Critère d'arrêt.** Au TP1, l'algorithme était arrêté lorsque $\|\nabla f(x_k)\|$ devenait petit. Ce critère n'est plus adapté en présence de contraintes : en général, $\nabla f(x^*) \neq 0$ à la solution (par exemple, dans la partie I, le gradient de $f$ ne s'annule pas sur le pavé). On utilise à la place l'**application gradient**
$$
    G_s(x) = \frac{1}{s}\Big(x - P_C\big(x - s \nabla f(x)\big)\Big).
$$
Lorsque $C = \mathbb{R}^n$, on retrouve $G_s(x) = \nabla f(x)$. Le critère d'arrêt devient
$$
    \frac{\| G_s(x_k) \|}{\| G_s(x_0) \|} < \eta.
$$

On peut montrer que $G_s(x) = 0$ si et seulement si $x$ est solution du problème.

**Todo :** Coder la fonction `gradient_projete(x0, proj, epsilon, nit, eta)` qui implémente l'algorithme du gradient projeté pour minimiser $R$ sur un ensemble $C$ dont la projection est donnée par la fonction `proj` :
* partir de $P_C(x_0)$ ;
* utiliser le pas $s = 1/L$ avec $L = 8/\epsilon$ ;
* arrêter l'algorithme lorsque $\| G_s(x_k) \| / \| G_s(x_0) \| < \eta$ ou après `nit` itérations ;
* stocker à chaque itération $R(x_k)$, $\|\nabla R(x_k)\|$ et $\|G_s(x_k)\|$ dans les listes `listcf`, `list_grad` et `list_gm`.

Cette fonction sera réutilisée dans la suite du TP.

In [ ]:
def gradient_projete(x0, proj, epsilon, nit=5000, eta=1e-2):
    L = 8 / epsilon
    step = 1 / L
    x = proj(x0)
    listcf, list_grad, list_gm = [], [], []

    ### à coder

    return x, listcf, list_grad, list_gm

**Todo :** Tester la méthode avec $\epsilon = 0.01$, $\eta = 10^{-2}$ et au plus 5000 itérations. Comme initialisation, on remplace les pixels manquants par la moyenne des pixels connus.

Le code suivant affiche les images, le PSNR, les courbes de convergence, le nombre d'itérations et le temps de calcul.

In [ ]:
epsilon = 0.01
x0 = np.where(mask, v, np.mean(v[mask]))
proj = lambda x: proj_inpainting(x, mask, v)

start = time.time()
x, listcf, list_grad, list_gm = gradient_projete(x0, proj, epsilon, nit=5000, eta=1e-2)
end = time.time()

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(v, cmap='gray'), axs[0].set_title('v')
axs[1].imshow(x0, cmap='gray'), axs[1].set_title('x0 : %.2fdB' % psnr(u, x0))
axs[2].imshow(x, cmap='gray'), axs[2].set_title('gradient projeté : %.2fdB' % psnr(u, x))
plt.show()

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].semilogy(listcf), axs[0].set_title('$R(x_k)$')
axs[1].semilogy(list_grad, label=r'$\|\nabla R(x_k)\|$')
axs[1].semilogy(list_gm, label=r'$\|G_s(x_k)\|$')
axs[1].legend()
plt.show()

nb_it = len(listcf)
print('Psnr inpainted: %.2fdB' % (psnr(u, x)))
print('Number of iterations: %d' % (nb_it))
print('Elapsed time: %.2f s' % (end - start))
print('Time spent per iteration: %.2fms' % (1000*(end - start)/nb_it))

**Todo :** Que se passerait-il si l'on oubliait l'étape de projection (descente de gradient classique sur $R$, en partant de $x_0$) ? Quels sont les minimiseurs de $R$ sur $\mathbb{R}^{M\times N}$ ? Vérifier numériquement.

In [ ]:
### à coder

**Todo :** Étudier l'influence de la proportion `p` de pixels manquants ($p \in \{0.3, 0.5, 0.7, 0.9\}$), puis celle de $\epsilon$ ($\epsilon \in \{10^{-1}, 10^{-2}, 10^{-3}\}$) sur le PSNR, le nombre d'itérations et le temps de calcul. Commenter.

In [ ]:
### à coder

## III. Débruitage sous contrainte : principe de Morozov

Au TP1, nous avons débruité une image $v = u + n$, $n \sim \mathcal{N}(0, \sigma^2 \textrm{Id})$, en minimisant
$$
    f_\lambda(u) = \frac{1}{2}\|u - v\|_2^2 + \lambda R(u).
$$
Le choix du paramètre $\lambda$ est délicat : il n'a pas d'interprétation physique simple. En revanche, le niveau de bruit $\sigma$ est souvent connu (ou facilement estimé). Comme $\mathbb{E}\|n\|_2^2 = MN\sigma^2$, et que $\|n\|_2$ se concentre fortement autour de sa moyenne lorsque $MN$ est grand, l'image $u$ que l'on cherche vérifie
$$
    \| u - v \|_2 \approx \sigma\sqrt{MN}.
$$
Le **principe de Morozov** consiste à chercher, parmi les images compatibles avec les observations au niveau de bruit près, la plus régulière :
$$
    \min_{u} R(u) \quad \textrm{s.c.} \quad \|u - v\|_2 \leq r, \qquad r = \tau \sigma \sqrt{MN},
$$
où $\tau \approx 1$. L'ensemble des contraintes est la boule $B(v, r)$, dont la projection a été codée à la partie I.

**Todo :** Générer une réalisation du bruit avec $\sigma = 0.05$ et vérifier numériquement que $\|n\|_2 \approx \sigma\sqrt{MN}$.

In [ ]:
np.random.seed(1)
sigma = 0.05
n = sigma * np.random.randn(M, N)
v = u + n

### à coder

**Todo :** Résoudre le problème avec la fonction `gradient_projete`, pour $\tau = 1$, $\epsilon = 0.01$, $\eta = 10^{-3}$ et $x_0 = v$. Afficher le PSNR, l'image débruitée et les courbes de convergence. La contrainte est-elle active à la solution ?

In [ ]:
epsilon = 0.01
tau = 1.
radius = tau * sigma * np.sqrt(M * N)

### à coder

**Todo :** Faire varier $\tau$ entre 0.6 et 1.4 et tracer le PSNR en fonction de $\tau$. Commenter.

In [ ]:
### à coder